<a href="https://colab.research.google.com/github/ivan-penta/proyecto_cidam/blob/main/proyecto_cidam_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Entorno Pyspark

In [1]:
# 1. Instalar Java 17 y PySpark
!apt-get update -qq
!apt-get install openjdk-17-jdk -y
!pip install pyspark==4.0.1 -q

# 2. Configurar variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# 3. Verificar instalación de Java
!java -version

# 4. Crear SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis Compranet Big Data") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

print("✅ SparkSession creada")
print("Versión:", spark.version)
print("App Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 11 not upgraded.
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
✅ SparkSession creada
Versión: 4.0.1
App Name: Analisis Compranet Big Data
Master: local[*]


### Cargar dataset

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr

spark = SparkSession.builder \
    .appName("Analisis Compranet Big Data") \
    .getOrCreate()

print("📂 Cargando dataset completo de CompraNet...")


data = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("compranet_historico.csv")

data = data.withColumn(
    "importe",
    expr("try_cast(importe as double)")
)

conteo_nulos = data.filter(col("importe").isNull()).count()
print(f"⚠️ Registros con formato inválido convertidos a NULL: {conteo_nulos}")

total_registros = data.count()
print(f"📊 Total de registros procesados: {total_registros:,}")

data.write.mode("overwrite").parquet("compranet_historico.parquet")
print("💾 Dataset completo guardado exitosamente en formato Parquet.")

📂 Cargando dataset completo de CompraNet...
⚠️ Registros con formato inválido convertidos a NULL: 5188
📊 Total de registros procesados: 2,356,612
💾 Dataset completo guardado exitosamente en formato Parquet.


### Limpieza de nulos

In [3]:
from pyspark.sql.functions import col

registros_antes = data.count()

data = data.dropna(subset=[
    "importe",
    "ff_fecha_inicio",
    "ff_fecha_fin"
])

registros_despues = data.count()
print(f"Registros antes de limpieza: {registros_antes:,}")
print(f"Registros después de limpieza: {registros_despues:,}")
print(f"Registros eliminados: {registros_antes - registros_despues:,}")

Registros antes de limpieza: 2,356,612
Registros después de limpieza: 2,347,566
Registros eliminados: 9,046


### Selección de variables

In [4]:
from pyspark.sql.functions import col

df = data.select(
    "codigo_contrato",
    "codigo_expediente",
    "proveedor",
    "contract_type",
    "tipo_contratacion",
    "tipo_expediente",
    "importe",
    "moneda",
    "fecha_inicio",
    "fecha_fin"
)

df.show(5)

+---------------+-----------------+--------------------+---------------+--------------------+--------------------+----------+------+--------------------+--------------------+
|codigo_contrato|codigo_expediente|           proveedor|  contract_type|   tipo_contratacion|     tipo_expediente|   importe|moneda|        fecha_inicio|           fecha_fin|
+---------------+-----------------+--------------------+---------------+--------------------+--------------------+----------+------+--------------------+--------------------+
|        2376191|          2161394|Equity Appraisal ...|    3.Servicios|           Servicios|05. Adjudicación ...|   89012.0|   MXN|2020-07-22 05:00:...|2020-08-27 04:59:...|
|             89|              348|Si Vale Mexico Sa...|ADQUISICIONES_0|No especificado p...|V20151220 12. Adj...| 5980292.7|   MXN|2010-12-06 06:00:...|2011-01-01 05:59:...|
|           1756|              399|Metlife Mexico Sa...|    SERVICIOS_1|No especificado p...|V20110525 01. Lic...| 3904647.0|

### Conversión de fechas y variable año

In [5]:
from pyspark.sql.functions import to_timestamp, year, col

df = df.withColumn("fecha_inicio", to_timestamp(col("fecha_inicio")))
df = df.withColumn("fecha_fin", to_timestamp(col("fecha_fin")))
df = df.filter(col("fecha_inicio").isNotNull())
df = df.withColumn("anio", year(col("fecha_inicio")))

df.select("fecha_inicio", "anio").show(5)

+-------------------+----+
|       fecha_inicio|anio|
+-------------------+----+
|2020-07-22 05:00:00|2020|
|2010-12-06 06:00:00|2010|
|2011-01-01 06:00:00|2011|
|2011-01-01 06:00:00|2011|
|2011-01-01 06:00:00|2011|
+-------------------+----+
only showing top 5 rows


### Filtro del periodo de estudio

In [6]:
from pyspark.sql.functions import to_date, lit

df = df.filter(
    (col("fecha_inicio") >= to_date(lit("2012-12-01"))) &
    (col("fecha_inicio") <= to_date(lit("2024-11-30")))
).filter(col("fecha_inicio").isNotNull())

print(f"Registros en periodo de estudio: {df.count():,}")

Registros en periodo de estudio: 2,052,702


### Creación de Variable Sexenio

In [7]:
from pyspark.sql.functions import when, to_timestamp, lit

df = df.withColumn(
    "sexenio",
    when(
        (col("fecha_inicio") >= to_timestamp(lit("2012-12-01 00:00:00"))) &
        (col("fecha_inicio") <  to_timestamp(lit("2018-12-01 00:00:00"))),
        "EPN"
    ).when(
        (col("fecha_inicio") >= to_timestamp(lit("2018-12-01 00:00:00"))) &
        (col("fecha_inicio") <  to_timestamp(lit("2024-12-01 00:00:00"))),
        "AMLO"
    )
)

df.groupBy("sexenio").count().show()

+-------+-------+
|sexenio|  count|
+-------+-------+
|    EPN|1270084|
|   AMLO| 782618|
+-------+-------+



### Ajuste por inflación

In [8]:
from pyspark.sql.functions import create_map, lit, col
from itertools import chain

#INPC base 2018
inpc = {
    2012: 47.23, 2013: 49.04, 2014: 51.10,
    2015: 52.36, 2016: 54.53, 2017: 57.71,
    2018: 61.20,
    2018: 65.28, 2020: 68.01, 2021: 72.58,
    2022: 80.95, 2023: 87.62, 2024: 92.10
}

inpc_map = create_map([lit(x) for x in chain(*inpc.items())])
INPC_BASE = 61.20

df = df.withColumn("inpc_anio", inpc_map[col("anio")]) \
  .withColumn(
      "importe_real",
      col("importe") * lit(INPC_BASE) / col("inpc_anio")
  )

  # verificar nulos tras la conversión
null_real = df.filter(col("importe_real").isNotNull())
print(f"Registros válidos tras ajuste: {df.count()}")

Registros válidos tras ajuste: 2052702


### Estadísticas descriptivas por sexenio

In [9]:
from pyspark.sql.functions import avg, sum, count, percentile_approx, round as spark_round, format_number

descriptivo = df.groupBy("sexenio").agg(
    count("*").alias("num_contratos"),
    spark_round(sum("importe_real"), 2).alias("importe_total"),
    spark_round(avg("importe_real"), 2).alias("importe_medio"),
    spark_round(
        percentile_approx("importe_real", 0.5).cast("double"), 2
    ).alias("importe_mediana")
)

descriptivo.select(
    "sexenio",
    format_number("num_contratos", 0).alias("contratos"),
    format_number("importe_total", 2).alias("total_real"),
    format_number("importe_medio", 2).alias("media_real"),
    format_number("importe_mediana", 2).alias("mediana_real")
).show(truncate=False)

+-------+---------+--------------------+------------+------------+
|sexenio|contratos|total_real          |media_real  |mediana_real|
+-------+---------+--------------------+------------+------------+
|EPN    |1,270,084|3,195,917,394,239.64|2,516,303.96|137,918.84  |
|AMLO   |782,618  |1,552,848,050,452.26|2,637,220.29|112,417.15  |
+-------+---------+--------------------+------------+------------+



### Modalidades de contratación por sexenio

In [10]:
from pyspark.sql.functions import count, round as spark_round, col

modalidades = df.groupBy("sexenio", "tipo_contratacion").count()

total_sexenio = df.groupBy("sexenio").agg(count("*").alias("total"))

modalidades_pct = modalidades.join(total_sexenio, "sexenio") \
    .withColumn(
        "porcentaje",
        spark_round(col("count") / col("total") * 100, 2)
    )

modalidades_pct.orderBy("sexenio", col("count").desc()).show(20, truncate=False)

+-------+--------------------------------+------+-------+----------+
|sexenio|tipo_contratacion               |count |total  |porcentaje|
+-------+--------------------------------+------+-------+----------+
|AMLO   |Adquisiciones                   |478476|782618 |61.14     |
|AMLO   |Servicios                       |242061|782618 |30.93     |
|AMLO   |Obra Pública                    |35050 |782618 |4.48      |
|AMLO   |Sin dato                        |13096 |782618 |1.67      |
|AMLO   |Servicios Relacionados con la OP|9526  |782618 |1.22      |
|AMLO   |Arrendamientos                  |4409  |782618 |0.56      |
|EPN    |Adquisiciones                   |647848|1270084|51.01     |
|EPN    |Servicios                       |440442|1270084|34.68     |
|EPN    |Obra Pública                    |143863|1270084|11.33     |
|EPN    |Servicios Relacionados con la OP|26888 |1270084|2.12      |
|EPN    |Arrendamientos                  |7953  |1270084|0.63      |
|EPN    |Sin dato                 

### Tipo de procedimiento

In [11]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "tipo_procedimiento",
    when(col("tipo_expediente").like("%Adjudicación Directa%"), "ADJUDICACION_DIRECTA")
    .when(col("tipo_expediente").like("%Invitación%"), "INVITACION_3")
    .when(col("tipo_expediente").like("%Licitación Pública%"), "LICITACION_PUBLICA")
    .otherwise("OTROS")
)

df.groupBy("tipo_procedimiento").count().orderBy(col("count").desc()).show()

+--------------------+-------+
|  tipo_procedimiento|  count|
+--------------------+-------+
|ADJUDICACION_DIRECTA|1530909|
|  LICITACION_PUBLICA| 271660|
|        INVITACION_3| 197839|
|               OTROS|  52294|
+--------------------+-------+



### Adjudicación directa por sexenio

In [12]:
from pyspark.sql.functions import count, round as spark_round, format_number, col

adj_sexenio  = df.groupBy("sexenio", "tipo_procedimiento").count()
total_sexenio = df.groupBy("sexenio").agg(count("*").alias("total"))

adj_pct = adj_sexenio.join(total_sexenio, "sexenio") \
    .withColumn(
        "porcentaje",
        format_number(col("count") / col("total") * 100, 2)
    )

adj_pct.orderBy("sexenio").show(20)

+-------+--------------------+------+-------+----------+
|sexenio|  tipo_procedimiento| count|  total|porcentaje|
+-------+--------------------+------+-------+----------+
|   AMLO|ADJUDICACION_DIRECTA|600200| 782618|     76.69|
|   AMLO|  LICITACION_PUBLICA| 90571| 782618|     11.57|
|   AMLO|               OTROS| 38776| 782618|      4.95|
|   AMLO|        INVITACION_3| 53071| 782618|      6.78|
|    EPN|ADJUDICACION_DIRECTA|930709|1270084|     73.28|
|    EPN|  LICITACION_PUBLICA|181089|1270084|     14.26|
|    EPN|        INVITACION_3|144768|1270084|     11.40|
|    EPN|               OTROS| 13518|1270084|      1.06|
+-------+--------------------+------+-------+----------+



### Duración promedio de contratos

In [13]:
from pyspark.sql.functions import datediff, avg, round as spark_round

df = df.withColumn(
    "duracion_dias",
    datediff(col("fecha_fin"), col("fecha_inicio"))
)

dura_sexenio = df.groupBy("sexenio").agg(
    spark_round(avg("duracion_dias"), 2).alias("duracion_promedio")
)

dura_sexenio.show()

+-------+-----------------+
|sexenio|duracion_promedio|
+-------+-----------------+
|    EPN|           111.29|
|   AMLO|           122.62|
+-------+-----------------+



### Top proveedores por sexenio y HHI

In [14]:
from pyspark.sql.functions import sum, col, format_number, pow, round as spark_round
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Top 10 por sexenio
proveedores_monto = df.groupBy("sexenio", "proveedor").agg(
    sum("importe_real").alias("monto_total")
)

window = Window.partitionBy("sexenio").orderBy(col("monto_total").desc())

proveedores_monto.withColumn("rank", row_number().over(window)) \
    .filter(col("rank") <= 10) \
    .select(
        "sexenio", "proveedor",
        format_number("monto_total", 2).alias("monto_total"),
        "rank"
    ).show(truncate=False)

# HHI
gasto_total = df.groupBy("sexenio").agg(sum("importe_real").alias("gasto_total"))

participacion = proveedores_monto.join(gasto_total, "sexenio") \
    .withColumn("s_i", col("monto_total") / col("gasto_total"))

hhi = participacion.withColumn("s2", pow(col("s_i"), 2)) \
    .groupBy("sexenio").agg(sum("s2").alias("HHI"))

print("\nÍndice de Herfindahl-Hirschman (HHI):")
hhi.show()
print("Referencia: HHI < 0.01 baja concentración | 0.01-0.18 moderada | > 0.18 alta")

+-------+--------------------------------------------------------------------+------------------+----+
|sexenio|proveedor                                                           |monto_total       |rank|
+-------+--------------------------------------------------------------------+------------------+----+
|AMLO   |Toka Internacional S a P I de Cv                                    |40,651,506,915.81 |1   |
|AMLO   |Alstom Transport Mexico Sa de Cv                                    |26,578,286,394.23 |2   |
|AMLO   |Ica Constructora Sa de Cv                                           |24,898,628,876.81 |3   |
|AMLO   |Currie & Brown - Mexico Sa de Cv                                    |21,745,478,093.14 |4   |
|AMLO   |Laboratorios de Biologicos y Reactivos de Mexico Sa de Cv           |14,529,756,377.69 |5   |
|AMLO   |Laboratorios Pisa S.A. de C.V.                                      |14,423,368,629.26 |6   |
|AMLO   |Operadora Cicsa Sa de Cv                                        

### Prueba *t* y *Mann-Whitney* para importe y duración

In [17]:
from scipy.stats import ttest_ind, mannwhitneyu
import pandas as pd

pdf = df.select("sexenio", "importe_real", "duracion_dias").dropna().toPandas()

epn = pdf[pdf["sexenio"] == "EPN"]
amlo = pdf[pdf["sexenio"] == "AMLO"]

print("Pruebas para importe real (año base 2018)")

# Prueba *t* de Welch (no se asume varianzas iguales)

t_stat, p_t = ttest_ind(epn["importe_real"], amlo["importe_real"], equal_var = False)
print(f"\nPrueba t de Welch:")
print(f"  t = {t_stat:.4f}  | p-value = {p_t:.2e}")
print(f" {'Se rechaza H0' if p_t < 0.05 else 'No se rechaza H0'} (α=0.05)")

# Prueba Mann-Whitney U (no paramétrica)

u_stat, p_mw = mannwhitneyu(epn["importe_real"], amlo["importe_real"], alternative="two-sided")
print(f"\nPrueba Mann-Whitney U:")
print(f"  U = {u_stat:.4f}  | p-value = {p_mw:.2e}")
print(f" {'Se rechaza H0' if p_mw < 0.05 else 'No se rechaza H0'} (α=0.05)")

print("\nPrueba para duración de contratos")

t_d, p_td =ttest_ind(epn["duracion_dias"], amlo["duracion_dias"], equal_var = False)
print(f"\nPrueba t de Welch:")
print(f"  t = {t_d:.4f}  | p-value = {p_td:.2e}")
print(f" {'Se rechaza H0' if p_td < 0.05 else 'No se rechaza H0'} (α=0.05)")

u_d, p_mwd = mannwhitneyu(epn["duracion_dias"], amlo["duracion_dias"], alternative="two-sided")

print(f"\nPrueba Mann-Whitney U:")
print(f"  U = {u_d:.4f}  | p-value = {p_mwd:.2e}")
print(f"  {'Se rechaza H0' if p_mwd < 0.05 else 'No se rechaza H0'} (α=0.05)")

Pruebas para importe real (año base 2018)

Prueba t de Welch:
  t = -0.8973  | p-value = 3.70e-01
 No se rechaza H0 (α=0.05)

Prueba Mann-Whitney U:
  U = 402448041841.0000  | p-value = 0.00e+00
 Se rechaza H0 (α=0.05)

Prueba para duración de contratos

Prueba t de Welch:
  t = -36.2958  | p-value = 2.77e-288
 Se rechaza H0 (α=0.05)

Prueba Mann-Whitney U:
  U = 372626087660.0000  | p-value = 1.28e-04
  Se rechaza H0 (α=0.05)


### Series de tiempo

In [ ]:
from pyspark.sql.functions import month, sum, count, col, format_number, lpad, concat_ws
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

